# 0 · Setup — auto-detect everything + enter your HF token once

Role, region, and the S3 bucket (SageMaker's default bucket) are **auto-detected**. Your Hugging Face token is entered **once here** like a password (`getpass`) and reused by every job.

This writes `summ_config.json` for notebooks 01–04.

> RLAIF note: the GRPO reward and the evaluation both call a **Bedrock** judge. The job role must allow `bedrock:InvokeModel` on the judge models. The **eval judge is a different model from the train judge** on purpose — so GRPO can't win by pleasing its own judge.

In [ ]:
# pip install -r ../requirements-launch.txt

In [ ]:
import json, time
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
region = sess.boto_region_name
try:
    role = get_execution_role()
except Exception as e:
    raise RuntimeError('Could not auto-detect an execution role. Run in a SageMaker '
                       f'notebook or set role manually. Original error: {e}')
bucket = sess.default_bucket()
try:
    account = sess.account_id()
except Exception:
    import boto3
    account = boto3.client('sts', region_name=region).get_caller_identity()['Account']
run_id = time.strftime('%Y%m%d-%H%M%S')
S3_BASE = f's3://{bucket}/summ-rlaif/{run_id}'
print('region :', region)
print('role   :', role)
print('bucket :', bucket, '(SageMaker default)')
print('run    :', S3_BASE)

### Enter your Hugging Face token (hidden input)

In [ ]:
import getpass, os
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face token (input hidden): ').strip()
assert HF_TOKEN, 'A Hugging Face token is required.'
from huggingface_hub import login, whoami
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF token OK for user:', whoami(token=HF_TOKEN)['name'])

### Choices you may want to change (working defaults; not placeholders)

> A **smaller base** (e.g. `Qwen/Qwen3.5-0.8B`) leaves SFT un-saturated, so GRPO has room to show value — same lesson as the tool-calling project.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3.5-4B'          # try 'Qwen/Qwen3.5-0.8B' for more GRPO headroom
GPU_INSTANCE = 'ml.p4de.24xlarge'       # 8x A100 80GB; p5.48xlarge also fine
CPU_INSTANCE = 'ml.m5.2xlarge'          # data-prep step only
GPU_IMAGE_URI = ''                      # '' -> resolve a stock PyTorch DLC

# RLAIF judges — EVAL MUST DIFFER FROM TRAIN.
JUDGE_BACKEND = 'bedrock'               # 'bedrock' | 'local' | 'mock'
TRAIN_JUDGE_MODEL = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'
EVAL_JUDGE_MODEL  = 'openai.gpt-oss-120b-1:0'  # OpenAI family — fully independent of the Sonnet train judge AND the Qwen policy

GPUS_PER_INSTANCE = {
    'ml.p4d.24xlarge': 8, 'ml.p4de.24xlarge': 8, 'ml.p5.48xlarge': 8,
    'ml.g5.12xlarge': 4, 'ml.g5.48xlarge': 8, 'ml.g6e.12xlarge': 4, 'ml.g6e.48xlarge': 8,
}
NUM_GPUS = GPUS_PER_INSTANCE.get(GPU_INSTANCE, 8)
print(f'{GPU_INSTANCE} -> NUM_GPUS={NUM_GPUS}')
assert EVAL_JUDGE_MODEL != TRAIN_JUDGE_MODEL or JUDGE_BACKEND == 'mock', \
    'Eval judge must differ from train judge (else GRPO can game its own judge).'

### Write the config

In [ ]:
cfg = {
    'region': region, 'role': role, 'account': account, 'hf_token': HF_TOKEN,
    'gpu_instance': GPU_INSTANCE, 'cpu_instance': CPU_INSTANCE,
    'num_gpus': NUM_GPUS, 'pt_version': '2.7', 'gpu_image_uri': GPU_IMAGE_URI,
    'keep_alive': 1800,   # warm-pool seconds after each job (0 disables; needs warm-pool quota; 3600 = API max)
    'base_model': BASE_MODEL,
    'tldr_dataset': 'CarperAI/openai_summarize_tldr',
    'n_train': 8000, 'n_grpo': 2000, 'n_eval': 400, 'n_pairwise': 200,
    'seq_len': 1024, 'seed': 42, 'enable_thinking': 0, 'attn_impl': 'flash_attention_2',
    'use_lora': 1, 'lora_r': 16, 'lora_alpha': 32,
    'sft_epochs': 1, 'sft_lr': '2e-4', 'sft_bs': 4, 'sft_grad_accum': 4,
    'grpo_epochs': 1, 'grpo_lr': '5e-6', 'grpo_bs': 8, 'grpo_grad_accum': 4,
    'num_generations': 8, 'grpo_beta': 0.01, 'grpo_temperature': 1.0,
    'max_completion_length': 128,
    'judge_backend': JUDGE_BACKEND, 'train_judge_model': TRAIN_JUDGE_MODEL,
    'eval_judge_model': EVAL_JUDGE_MODEL, 'judge_max_concurrency': 8,
    'reward_weights': '0.30,0.40,0.15,0.15',
    'reward_mode': 'guarded', 'guard_weights': '0.45,0.30,0.25', 'faith_gate': 0.40, 'gate_mult': 0.30, 'compare_against': 'sft',
    'eval_base': 1, 'eval_max_new_tokens': 128, 'eval_batch_size': 16,
    's3_base': S3_BASE, 'data_s3': S3_BASE + '/data',
    'sft_model_s3': S3_BASE + '/sft_model', 'grpo_model_s3': S3_BASE + '/grpo_model',
    'results_s3': S3_BASE + '/results',
}
_glob = cfg['grpo_bs'] * cfg['num_gpus'] * cfg['grpo_grad_accum']
assert _glob % cfg['num_generations'] == 0, (
    f"GRPO batch {_glob} not divisible by num_generations {cfg['num_generations']}.")
json.dump(cfg, open('summ_config.json', 'w'), indent=2)
print(json.dumps({**cfg, 'hf_token': '***'}, indent=2))
print('\nwrote summ_config.json (token redacted above)')

**Next:** `01_prepare.ipynb`.